# 🎬 OpenSource Clipping Studio — Backend Server

Run the Web Studio backend on a Kaggle or Colab GPU and connect from the browser dashboard.

**Runtime:** Linux x86_64, Internet enabled. A Python 3.13 notebook kernel is supported by preparing a separate CPython 3.12 for the application; the kernel itself is not changed.

Setup uses an isolated virtual environment and starts FastAPI in a fresh process. The public URL appears only after actual renderer imports and FFmpeg pass readiness checks. Keep the Start Backend Server cell running while using Studio.

Gemini and NVIDIA NIM use cloud API keys. To also run Gemma 4 Q4 in this session, enable the optional `START_LOCAL_VLLM` switch below. Otherwise Studio can connect to an existing vLLM server using the configured endpoint.

See `docs/KAGGLE_STUDIO.md` in the repository for troubleshooting and dependency details.

## 1. Setup Project

Clone into a named directory without deleting existing files. Existing checkouts are preserved, not automatically updated. Ensure your checkout contains this notebook's matching `scripts/kaggle_studio.py`, `scripts/kaggle_python.py` and requirements files before running setup.

In [1]:
# Clone repository without deleting the notebook working directory.
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/NaufalRizqullah/opensource-clipping.git"
workspace = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path("/content")
if not workspace.is_dir():
    raise RuntimeError("Run this notebook in Kaggle or Colab, not on a local PC.")
PROJECT_DIR = Path.cwd() if (Path.cwd() / "scripts/kaggle_studio.py").is_file() else workspace / "opensource-clipping"
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
elif not (PROJECT_DIR / ".git").is_dir():
    raise RuntimeError("Project directory already exists and is not a checkout. Select a different PROJECT_DIR.")
required = ["scripts/kaggle_storage.py", "scripts/kaggle_studio.py", "scripts/kaggle_python.py", "requirements-kaggle.txt"]
if any(not (PROJECT_DIR / path).is_file() for path in required):
    raise RuntimeError("Update this checkout to the revision containing the Kaggle Studio fixes, then rerun setup.")
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Project checkout ready. Existing outputs are retained.")

Cloning into '/kaggle/working/opensource-clipping'...


Project checkout ready. Existing outputs are retained.


## 1b. Storage and Python 3.12

Temporary mode keeps Python, installed packages and temporary downloads outside the saved output. Use a dedicated writable RUNTIME_ROOT on another filesystem if the reported free space is low. The checkout mode reuses the previous environment locations. Save existing clips and logs before starting a fresh session.

Select the CPU Torch profile for a smaller Studio install. Use Whisper CPU/int8 or subtitles with that profile. Keep cuda for Studio GPU features. Local vLLM has its own GPU environment.

The bootstrap reuses CPython 3.12 when available or downloads it for Studio. Rerun after a kernel restart.

In [2]:
from scripts.kaggle_python import bootstrap_environment, ensure_python

from scripts.kaggle_storage import check_storage, prepare_storage

STORAGE_MODE = "temporary"  # "checkout" uses the original .cache paths
RUNTIME_ROOT = None  # Optional absolute path on a writable scratch disk
MIN_OUTPUT_GIB = 1.0  # Increase to the space needed for your source and clips
STUDIO_TORCH_PROFILE = "cuda"  # "cpu" saves disk; choose Whisper CPU/int8 or subtitles
if STUDIO_TORCH_PROFILE not in {"cuda", "cpu"}:
    raise ValueError("Choose cuda or cpu for STUDIO_TORCH_PROFILE.")

storage = prepare_storage(PROJECT_DIR, workspace, mode=STORAGE_MODE, runtime_root=RUNTIME_ROOT)
check_storage(storage, MIN_OUTPUT_GIB)
cache_env = storage.cache_environment()
PYTHON_CACHE = storage.python_cache
BOOTSTRAP_PYTHON = ensure_python(PYTHON_CACHE)
print(f"Notebook kernel: Python {sys.version_info.major}.{sys.version_info.minor}")
print(f"Setup Python: {BOOTSTRAP_PYTHON}")
subprocess.run(
    [str(BOOTSTRAP_PYTHON), "--version"],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)


Runtime: /kaggle/temp/opensource-clipping-abde345f097e (2514.55 GiB free)
Output:  /kaggle/working (19.35 GiB free, 1 GiB requested for jobs)
Notebook kernel: Python 3.13
Setup Python: /usr/bin/python3.12
Python 3.12.3


CompletedProcess(args=['/usr/bin/python3.12', '--version'], returncode=0)

## 1c. Install Studio dependencies

The application environment uses Python 3.12 selected above. Keep it separate from any vLLM server environment. A failed install or dependency check stops setup.

In [3]:
# Fail fast and keep installer output visible. Do not pip-install into this kernel.
STUDIO_ENV = (storage.studio_env if STUDIO_TORCH_PROFILE == "cuda"
              else storage.root / "kaggle-studio-cpu" / "venv")
STUDIO_PYTHON = STUDIO_ENV / "bin" / "python"
subprocess.run(
    [str(BOOTSTRAP_PYTHON), str(PROJECT_DIR / "scripts/kaggle_studio.py"), "setup", "--venv", str(STUDIO_ENV),
     "--torch-profile", STUDIO_TORCH_PROFILE],
    cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
)

Diagnostic log: /kaggle/working/opensource-clipping/logs/kaggle-studio.jsonl
Creating isolated Kaggle Studio environment...
Bootstrapping pip inside the isolated environment...
Host pip unavailable; using the official PyPA bootstrap in the target environment...


/usr/bin/python3.12: No module named pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.4 MB/s  0:00:00
pip 26.3 from /kaggle/temp/opensource-clipping-abde345f097e/kaggle-studio/venv/lib/python3.12/site-packages/pip (python 3.12)
INFO: pip is looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of google-api-core to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is c

CompletedProcess(args=['/usr/bin/python3.12', '/kaggle/working/opensource-clipping/scripts/kaggle_studio.py', 'setup', '--venv', '/kaggle/temp/opensource-clipping-abde345f097e/kaggle-studio/venv', '--torch-profile', 'cuda'], returncode=0)

In [ ]:
# Install FFmpeg only if missing; failed system commands stop this cell.
import shutil
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "ffmpeg"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("FFmpeg ready.")

## 2. Configure API Keys

In Kaggle **Add-ons → Secrets**, create the exact label `NGROK_AUTHTOKEN` and enable/attach it to this notebook. A secret saved in your account but not attached to the running notebook cannot be read. In Colab, grant notebook access to the matching userdata secret.

Configure `GOOGLE_API_KEY` for Gemini or `NVIDIA_API_KEY` for NVIDIA. vLLM does not require either cloud key; `VLLM_API_KEY` is optional authentication for an existing inference server. `PEXELS_API_KEY` and `HF_TOKEN` support optional features. After adding or changing a secret, rerun this cell before starting the server. Setting a Python variable alone does not export it to subprocesses.

Keys are passed through the environment only. Their values are never printed or written to `.env`. The managed server ignores existing `.env` files.

In [7]:
import os

secret_readers = []
try:
    from kaggle_secrets import UserSecretsClient
    secret_readers.append(UserSecretsClient().get_secret)
except ImportError:
    pass
except Exception as exc:
    print(f"Kaggle Secrets unavailable ({type(exc).__name__}); check notebook access.")
try:
    from google.colab import userdata
    secret_readers.append(userdata.get)
except ImportError:
    pass

for name in ("GOOGLE_API_KEY", "NVIDIA_API_KEY", "PEXELS_API_KEY", "HF_TOKEN", "NGROK_AUTHTOKEN", "VLLM_API_KEY"):
    value = os.environ.get(name, "").strip()
    for read_secret in secret_readers:
        if value:
            break
        try:
            value = (read_secret(name) or "").strip()
        except Exception as exc:
            if name == "NGROK_AUTHTOKEN":
                print(f"NGROK_AUTHTOKEN could not be read ({type(exc).__name__}); check its exact label and notebook attachment.")
    if value:
        os.environ[name] = value
    print(f"{name}: {'configured' if value else 'not configured'}")
del value

if not os.environ.get("NGROK_AUTHTOKEN", "").strip():
    raise RuntimeError("NGROK_AUTHTOKEN is not available. Attach this exact secret label to the notebook, then rerun this cell before starting the server.")


GOOGLE_API_KEY: configured
NVIDIA_API_KEY: not configured
PEXELS_API_KEY: not configured
HF_TOKEN: not configured
NGROK_AUTHTOKEN: configured
VLLM_API_KEY: not configured


In [8]:
# ──────────────────────────────────────────────────────────────────────
# YouTube Cookies Setup (Optional) - Solves yt-dlp 403 Forbidden errors
# ──────────────────────────────────────────────────────────────────────
import os
import shutil

COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
else:
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.


## 3. Optional vLLM / Gemma 4 Q4

Leave `START_LOCAL_VLLM=False` for Gemini/NVIDIA or to connect Studio to an already-running vLLM server. In that case, configure `VLLM_BASE_URL`, `VLLM_MODEL` and the matching optional `VLLM_API_KEY` through the environment/configuration cell; no inference server is installed or started here.

Set `START_LOCAL_VLLM=True` to run the pinned Gemma 4 Q4 preset in this notebook. It uses a separate venv, one supported GPU (index 0 by default), port 8001 and the existing T4 attention workaround. Two T4 cards are not pooled into one GPU. CPU-only and P100 sessions are unsupported by this local preset. Studio remains on port 8000.

The optional cells below install/start the local model and evaluates a short transcript before Studio opens. The earlier T4 run passed that eval; full video processing and clip quality still need validation. For this preset choose **Whisper CPU/int8 or YouTube subtitles** in Studio. See `docs/KAGGLE_VLLM.md` for details.

Before rerunning the combined local setup/start/eval cell, stop the managed vLLM server using the Stop local vLLM cell below. Setup deliberately refuses to modify an environment while its model server is running. Keep the API key unchanged until that server is stopped.

Setup pins CPU TorchCodec for media decoding while vLLM inference remains on the GPU. It checks native imports before model startup; failures are saved to `logs/kaggle-vllm-import.log`. For the CUDA 13 TorchCodec error, update the checkout and rerun Setup before Start. The helper automatically selects the parser for official Gemma 4 and Qwen3 models; see the guide for model overrides.

In [9]:
# Opt in only when you want this notebook to host a local LLM as well as Studio.
START_LOCAL_VLLM = False
VLLM_ENV = storage.vllm_env
GPU_INDEX = 0
VLLM_PORT = 8001
VLLM_MODEL = os.environ.get("VLLM_MODEL", "google/gemma-4-E2B-it-qat-w4a16-ct")
CONTEXT_LIMIT = int(os.environ.get("VLLM_CONTEXT_LIMIT", "16384"))
OUTPUT_TOKENS = int(os.environ.get("VLLM_MAX_TOKENS", "4096"))
# Optional Qwen3 example: uncomment all three lines together (validate in Kaggle).
# VLLM_MODEL = "Qwen/Qwen3-4B-AWQ"
# CONTEXT_LIMIT = 8192
# OUTPUT_TOKENS = 2048
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://127.0.0.1:8001")

if START_LOCAL_VLLM:
    from secrets import token_urlsafe
    VLLM_BASE_URL = f"http://127.0.0.1:{VLLM_PORT}"
    if not os.environ.get("VLLM_API_KEY", "").strip():
        os.environ["VLLM_API_KEY"] = token_urlsafe(32)

os.environ.update({
    "VLLM_BASE_URL": VLLM_BASE_URL,
    "VLLM_MODEL": VLLM_MODEL,
    "VLLM_CONTEXT_LIMIT": str(CONTEXT_LIMIT),
    "VLLM_MAX_TOKENS": str(OUTPUT_TOKENS),
})
os.environ.setdefault("VLLM_TIMEOUT", "900")
print("Provider configuration ready. Keys are not displayed.")


Provider configuration ready. Keys are not displayed.


In [10]:
from scripts.kaggle_studio import runtime_environment

if START_LOCAL_VLLM:
    required = ["scripts/kaggle_vllm.py", "scripts/kaggle_vllm_compat.py",
                "scripts/eval_local_llm.py", "requirements-vllm-kaggle.txt"]
    if any(not (PROJECT_DIR / path).is_file() for path in required):
        raise RuntimeError("Update this checkout to include the vLLM helpers before continuing.")
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "preflight",
         "--gpu-index", str(GPU_INDEX)],
        cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
    )
    check_storage(storage, MIN_OUTPUT_GIB)
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "setup", "--venv", str(VLLM_ENV)],
        cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
    )
    check_storage(storage, MIN_OUTPUT_GIB)
    subprocess.run(
        [str(BOOTSTRAP_PYTHON), "scripts/kaggle_vllm.py", "start",
         "--venv", str(VLLM_ENV), "--gpu-index", str(GPU_INDEX),
         "--model", VLLM_MODEL, "--port", str(VLLM_PORT),
         "--context", str(CONTEXT_LIMIT), "--gpu-memory-utilization", "0.85"],
        cwd=PROJECT_DIR, env=bootstrap_environment(PYTHON_CACHE) | cache_env, check=True,
    )
    EVAL_OUTPUT = workspace / "vllm-eval-short.json"
    evaluation = subprocess.run(
        [str(STUDIO_PYTHON), "scripts/eval_local_llm.py",
         "--transcript", "tests/fixtures/eval_transcript_short.txt",
         "--clips", "1", "--output", str(EVAL_OUTPUT)],
        cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV) | cache_env, check=False,
    )
    print(f"Eval report: {EVAL_OUTPUT}")
    if evaluation.returncode:
        server_log = PROJECT_DIR / "logs/kaggle-vllm.log"
        if server_log.is_file():
            print("\n".join(server_log.read_text(errors="replace").splitlines()[-120:]))
        raise RuntimeError("vLLM eval failed. Save the report and server log before restarting.")
else:
    print("Local vLLM setup skipped. Gemini/NVIDIA remain available; vLLM requires a running configured server.")


Local vLLM setup skipped. Gemini/NVIDIA remain available; vLLM requires a running configured server.


## 4. Start Backend Server + Tunnel

This cell launches a fresh Python process from the isolated environment. Startup verifies actual renderer imports before opening ngrok; allow several minutes for the first check.

Copy the **Public URL** only after **BACKEND SERVER READY** appears, then paste it into Studio → Connect. Select Google Gemini, NVIDIA NIM or vLLM under New Job → AI & Whisper → AI Provider. For the single-GPU local vLLM preset, start with one clip and Whisper Device CPU (int8), or YouTube subtitles. Keep this cell running. Stop the cell or use Studio's Stop Server button to close the backend and tunnel.

If setup or readiness fails, fix that error and rerun; a healthy HTTP server alone is not sufficient.

Flow diagnostics are saved automatically to `logs/kaggle-studio.jsonl` (setup/supervisor) and `logs/studio-api.jsonl` (API/worker). If startup or a job fails, download these files and any `.1`/`.2`/`.3` backups from Kaggle Files/Output. They contain stage names, correlation IDs and safe traceback locations, without credential values or raw provider responses.

In [15]:
from scripts.kaggle_studio import runtime_environment, stop_process

# The supervisor and Uvicorn both use the isolated interpreter, never this kernel.
check_storage(storage, MIN_OUTPUT_GIB)
supervisor = subprocess.Popen(
    [str(STUDIO_PYTHON), str(PROJECT_DIR / "scripts/kaggle_studio.py"), "start", "--venv", str(STUDIO_ENV)],
    cwd=PROJECT_DIR, env=runtime_environment(STUDIO_ENV) | cache_env, start_new_session=True,
)
try:
    return_code = supervisor.wait()
    if return_code not in (0, 130):
        raise RuntimeError(f"Studio server exited with code {return_code}. Check the startup diagnostics above.")
except KeyboardInterrupt:
    print("Stopping Studio backend and tunnel...")
finally:
    stop_process(supervisor)


Runtime: /kaggle/temp/opensource-clipping-abde345f097e (2504.34 GiB free)
Output:  /kaggle/working (19.13 GiB free, 1 GiB requested for jobs)
Diagnostic log: /kaggle/working/opensource-clipping/logs/kaggle-studio.jsonl
Waiting for FastAPI and renderer readiness...


INFO:     Started server process [589]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


🚀 OpenSource Clipping Studio — Backend starting...
Studio diagnostic log: /kaggle/working/opensource-clipping/logs/studio-api.jsonl
INFO:     127.0.0.1:36848 - "GET /api/ready HTTP/1.1" 200 OK

BACKEND SERVER READY
Public URL: https://possum-armoire-utter.ngrok-free.dev
Paste this URL into Studio > Connect. Keep this cell running.
INFO:     106.219.69.170:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/ready HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/ready HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/ready HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/jobs/b07f1a181bd2 HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/jobs HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "GET /api/health HTTP/1.1" 200 OK
INFO:     106.219.69.170:0 - "G

      Transkripsi:   1%|          | 5/405s [00:01<01:55]

INFO:     106.219.69.170:0 - "GET /api/jobs/4d3283b5ade9 HTTP/1.1" 200 OK


      Transkripsi:   8%|▊         | 31/405s [00:03<00:37]

... (25 outputs left out)

I0000 00:00:1795162111.917248     660 task_runner.cc:85] GPU suport is not available: INTERNAL: ; RET_CHECK failure (mediapipe/gpu/gl_context_egl.cc:84) egl_initializedUnable to initialize EGL
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1795162111.924396     856 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


🧠 Rank 1 Hook - Analisa wajah dimulai...
INFO:     106.219.69.170:0 - "GET /api/jobs/4d3283b5ade9 HTTP/1.1" 200 OK
⏳ Rank 1 Hook - Analisa wajah:   0%
⏳ Rank 1 Hook - Analisa wajah:   8%
⏳ Rank 1 Hook - Analisa wajah:  16%
⏳ Rank 1 Hook - Analisa wajah:  25%
⏳ Rank 1 Hook - Analisa wajah:  33%
⏳ Rank 1 Hook - Analisa wajah:  41%
⏳ Rank 1 Hook - Analisa wajah:  50%
INFO:     106.219.69.170:0 - "GET /api/jobs/4d3283b5ade9 HTTP/1.1" 200 OK
⏳ Rank 1 Hook - Analisa wajah:  58%
⏳ Rank 1 Hook - Analisa wajah:  66%
⏳ Rank 1 Hook - Analisa wajah:  75%
⏳ Rank 1 Hook - Analisa wajah:  83%
INFO:     106.219.69.170:0 - "GET /api/jobs/4d3283b5ade9 HTTP/1.1" 200 OK
⏳ Rank 1 Hook - Analisa wajah:  91%
⏳ Rank 1 Hook - Analisa wajah: 100%
🎬 Rank 1 Hook - Render frame dimulai...
⏳ Rank 1 Hook - Render frame:   0% | 00:00 / 00:03
⏳ Rank 1 Hook - Render frame:   1% | 00:00 / 00:03
⏳ Rank 1 Hook - Render frame:   2% | 00:00 / 00:03
⏳ Rank 1 Hook - Render frame:   4% | 00:00 / 00:03
INFO:     106.219.69.170:

INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [589]


👋 Backend shutting down...
Studio server stopped.


In [16]:
r = subprocess.run("ffmpeg -hide_banner -encoders | grep -i nvenc", shell=True, capture_output=True, text=True)
print(r.stdout or "NO nvenc encoders in this ffmpeg build")
print(subprocess.run("nvidia-smi -L", shell=True, capture_output=True, text=True).stdout)

 V....D av1_nvenc            NVIDIA NVENC av1 encoder (codec av1)
 V....D h264_nvenc           NVIDIA NVENC H.264 encoder (codec h264)
 V....D hevc_nvenc           NVIDIA NVENC hevc encoder (codec hevc)

GPU 0: Tesla T4 (UUID: GPU-714cb593-3b9f-9916-ec97-efd27d17ded0)
GPU 1: Tesla T4 (UUID: GPU-778ca916-2346-7ec0-6f00-2e46c5e7d9c6)



In [17]:
t = subprocess.run(
    "ffmpeg -y -hide_banner -f lavfi -i testsrc=duration=3:size=1920x1080:rate=30 "
    "-c:v h264_nvenc -preset p4 /tmp/nvenc_test.mp4",
    shell=True, capture_output=True, text=True)
print("NVENC WORKS" if t.returncode == 0 else t.stderr[-1000:])

NVENC WORKS


In [18]:
subprocess.run("grep -rnE 'libx264|nvenc|-c:v|vcodec|preset|crf' --include=*.py .", shell=True, cwd=PROJECT_DIR)

./tests/test_kaggle_vllm.py:34:def test_preset_pins_qat_checkpoint_and_constrains_single_gpu(tmp_path, monkeypatch):
./tests/test_worker.py:67:            detect_video_encoder=lambda *a, **k: "libx264", proses_klip=render,
./clipping/engine.py:27:    # Note: Using [vcodec!*=av01] to safely ensure it does not contain 'av01' anywhere.
./clipping/engine.py:28:    codec_filter = "[vcodec!*=av01]"
./clipping/engine.py:192:            "format": "bestvideo[vcodec^=h264]+bestaudio/best[vcodec^=h264]/best",
./clipping/engine.py:235:            v_codec = info.get("vcodec", "unknown")
./clipping/studio/edge_glow.py:140:        "-c:v", "libx264", "-preset", "ultrafast",
./clipping/studio/ffmpeg_utils.py:29:        name: Encoder name, for example `h264_nvenc`.
./clipping/studio/ffmpeg_utils.py:87:    nvenc_preset_fast = "p1"
./clipping/studio/ffmpeg_utils.py:88:    nvenc_preset_legacy = "fast"
./clipping/studio/ffmpeg_utils.py:89:    nvenc_cq = 25
./clipping/studio/ffmpeg_utils.py:90:    cpu_preset

CompletedProcess(args="grep -rnE 'libx264|nvenc|-c:v|vcodec|preset|crf' --include=*.py .", returncode=0)

In [19]:
subprocess.run("grep -rhE 'NVIDIA NVENC|Fallback ke CPU' logs/ studio-start.log 2>/dev/null | tail -5",
               shell=True, cwd=PROJECT_DIR)

CompletedProcess(args="grep -rhE 'NVIDIA NVENC|Fallback ke CPU' logs/ studio-start.log 2>/dev/null | tail -5", returncode=0)

In [12]:
r = subprocess.run([str(STUDIO_PYTHON), "-m", "pip", "install", "-U", "av"],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])

In [14]:
r = subprocess.run(
    [str(STUDIO_PYTHON), "-m", "pip", "install", "--only-binary=:all:", "av<16"],
    capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 73.4 MB/s  0:00:00
  Attempting uninstall: av
    Found existing installation: av 19.0.2
    Uninstalling av-19.0.2:
      Successfully uninstalled av-19.0.2
 


## 5. Stop optional local vLLM and collect logs

Run after Studio stops to free the GPU used by the optional local model. The helper only stops the vLLM process it manages. With `START_LOCAL_VLLM=False`, an existing external inference server is left alone. If model startup/eval failed earlier, this cell can also be run explicitly for cleanup. Model downloads, reports and logs are preserved.

In [ ]:
if START_LOCAL_VLLM:
    subprocess.run(
        [sys.executable, str(PROJECT_DIR / "scripts/kaggle_vllm.py"), "stop"],
        cwd=PROJECT_DIR, check=True,
    )


In [ ]:
# Run after stopping the server to locate saved diagnostic files.
log_dir = Path(os.environ.get("OSC_LOG_DIR", str(PROJECT_DIR / "logs")))
log_files = sorted(log_dir.glob("*.jsonl*")) if log_dir.is_dir() else []
if not log_files:
    print("No diagnostic files yet. Run setup/start with the updated checkout first.")
for log_file in log_files:
    print(f"{log_file} ({log_file.stat().st_size:,} bytes)")
for extra in (PROJECT_DIR / "logs/kaggle-vllm.log",
              PROJECT_DIR / "logs/kaggle-vllm-import.log", workspace / "vllm-eval-short.json"):
    if extra.is_file():
        print(f"{extra} ({extra.stat().st_size:,} bytes)")
print("Download these files from Kaggle Files/Output; include rotated backups if present.")
print("Review third-party vLLM console logs before sharing them.")
